In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

# Create necessary directories
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# ============ LOAD DATA ============
df = pd.read_csv('used_cars.csv')
print(f"Original dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Create a copy for processing
df_processed = df.copy()

# ============ STEP 1: HANDLE MISSING VALUES ============
print("\n" + "=" * 80)
print("STEP 1: HANDLING MISSING VALUES")
print("=" * 80)

# Strategy for different columns:
# 1. Drop rows where target (price) is missing
df_processed = df_processed.dropna(subset=['price'])
print(f"Dropped rows with missing price: {df.shape[0] - df_processed.shape[0]}")

# 2. Drop columns with >40% missing values
missing_pct = (df_processed.isnull().sum() / len(df_processed)) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
print(f"\nColumns to drop (>40% missing): {cols_to_drop}")
df_processed = df_processed.drop(columns=cols_to_drop)

# 3. Fill missing numerical values with median
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val:.2f}")

# 4. Fill missing categorical values with mode
categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        mode_val = df_processed[col].mode()[0]
        df_processed[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

print(f"\nMissing values after cleaning: {df_processed.isnull().sum().sum()}")

# ============ DATA TYPE CONVERSION & UNIT CLEANING ============
print("\n" + "=" * 80)
print("DATA TYPE CONVERSION & UNIT CLEANING")
print("=" * 80)

df_processed['price'] = df_processed['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
df_processed['milage'] = df_processed['milage'].astype(str).str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)
df_processed.rename(columns={'milage': 'Kilometers_Driven'}, inplace=True)
df_processed['engine'] = df_processed['engine'].astype(str).str.extract('(\d+\.?\d*)', expand=False).astype(float)
df_processed.rename(columns={'engine': 'Engine'}, inplace=True)

# ============ STEP 2: REMOVE DUPLICATES ============
df_processed = df_processed.drop_duplicates()

# ============ STEP 3: HANDLE OUTLIERS ============
def remove_outliers_iqr(data, columns, multiplier=1.5):
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        data = data[(data[col] >= Q1 - multiplier * IQR) & (data[col] <= Q3 + multiplier * IQR)]
    return data

outlier_cols = ['price', 'Kilometers_Driven', 'Engine']
df_processed = remove_outliers_iqr(df_processed, [c for c in outlier_cols if c in df_processed.columns])

# ============ STEP 4: CATEGORICAL ENCODING ============
cat_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
ordinal_mappings = {'Owner_Type': {'First': 0, 'Second': 1, 'Third': 2, 'Fourth': 3, 'Fourth+': 4}}
for col, mapping in ordinal_mappings.items():
    if col in df_processed.columns: df_processed[col] = df_processed[col].map(mapping)

nominal_cat_cols = [col for col in cat_cols if col not in ordinal_mappings]
df_processed = pd.get_dummies(df_processed, columns=nominal_cat_cols, drop_first=True)

# ============ STEP 5: FEATURE ENGINEERING ============
if 'model_year' in df_processed.columns:
    df_processed['Age'] = 2024 - df_processed['model_year']
    df_processed.drop(columns=['model_year'], inplace=True)

if 'price' in df_processed.columns and 'Kilometers_Driven' in df_processed.columns:
    df_processed['price_per_Km'] = df_processed['price'] / (df_processed['Kilometers_Driven'] + 1)

if 'Engine' in df_processed.columns:
    df_processed['Engine_Size_Category'] = pd.cut(df_processed['Engine'], bins=[0, 1000, 1500, np.inf], labels=['Small', 'Medium', 'Large'], right=False)
    df_processed = pd.get_dummies(df_processed, columns=['Engine_Size_Category'], drop_first=True)
    df_processed['Engine_Squared'] = df_processed['Engine'] ** 2

if 'Age' in df_processed.columns and 'Kilometers_Driven' in df_processed.columns:
    df_processed['Age_Km_Interaction'] = df_processed['Age'] * df_processed['Kilometers_Driven']

# ============ STEP 6: FEATURE SCALING ============
X = df_processed.drop('price', axis=1)
y = df_processed['price']
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features] = scaler.fit_transform(X[numeric_features])

# Save scaler and feature names
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X_train.columns.tolist() if 'X_train' in locals() else X.columns.tolist(), '../models/feature_names.pkl')

# ============ STEP 7: TRAIN-TEST SPLIT ============
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# ============ STEP 8: SAVE PROCESSED DATA ============
X_train.to_csv('../data/processed/X_train.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Preprocessing complete and files saved successfully.")

Original dataset shape: (4009, 12)
Columns: ['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']

STEP 1: HANDLING MISSING VALUES
Dropped rows with missing price: 0

Columns to drop (>40% missing): []
Filled fuel_type with mode: Gasoline
Filled accident with mode: None reported
Filled clean_title with mode: Yes

Missing values after cleaning: 0

DATA TYPE CONVERSION & UNIT CLEANING
Preprocessing complete and files saved successfully.
